# **Notebook 05 — Modelling: XGBoost**

## Objectives

* Compare baseline algorithms (Random Forest, Gradient Boosting, XGBoost)
* Perform hyperparameter optimisation with 8+ parameters and rationale
* Evaluate model on train and test sets (confusion matrix, classification report)
* Generate SHAP explainability artifacts
* Validate Hypothesis H4

## Inputs

* `outputs/v1/X_train_resampled.csv`, `outputs/v1/y_train_resampled.csv`
* `outputs/v1/X_test_engineered.csv`, `outputs/v1/y_test.csv`

## Outputs

* `outputs/v1/xgb_baseline.pkl` — baseline model
* `outputs/v2/fraud_model_optimized.pkl` — optimised model
* `outputs/v2/shap_explainer.pkl` — SHAP explainer
* `outputs/v2/optimal_threshold.json` — best threshold
* `outputs/v2/evaluation_report.json` — metrics summary

---
## Change working directory

In [1]:
import os

current_dir = os.getcwd()
if current_dir.endswith("notebooks"):
    os.chdir(os.path.dirname(current_dir))
print(f"Working directory: {os.getcwd()}")

Working directory: C:\Users\name\Desktop\All IN\Code Inst. Projects\credit-card-fraud-detection


In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib
import json

X_train = pd.read_csv("outputs/v1/X_train_resampled.csv")
y_train = pd.read_csv("outputs/v1/y_train_resampled.csv").squeeze()
X_test = pd.read_csv("outputs/v1/X_test_engineered.csv")
y_test = pd.read_csv("outputs/v1/y_test.csv").squeeze()

print(f"Train set: {X_train.shape} (SMOTE balanced)")
print(f"Test set:  {X_test.shape} (original distribution)")
print(f"Train class balance: {y_train.value_counts().to_dict()}")
print(f"Test fraud cases:    {y_test.sum()}")

Train set: (453204, 38) (SMOTE balanced)
Test set:  (56746, 38) (original distribution)
Train class balance: {0: 226602, 1: 226602}
Test fraud cases:    95


---
## 1. Algorithm Comparison

Comparing three ensemble classifiers to select the best base algorithm before hyperparameter tuning. Using 5-fold stratified cross-validation for fair comparison.

In [3]:
! pip install xgboost

In [ ]:
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from xgboost import XGBClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score

models = {
    "Random Forest": RandomForestClassifier(
        n_estimators=100, random_state=42, class_weight='balanced'
    ),
    "Gradient Boosting": GradientBoostingClassifier(
        n_estimators=100, random_state=42
    ),
    "XGBoost": XGBClassifier(
        n_estimators=100, random_state=42,
        eval_metric='logloss', use_label_encoder=False
    )
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print("Algorithm Comparison (5-fold Stratified CV)")
print("=" * 60)

comparison_results = {}
for name, model in models.items():
    f1_scores = cross_val_score(model, X_train, y_train, cv=cv, scoring='f1')
    pr_scores = cross_val_score(model, X_train, y_train, cv=cv, scoring='precision')
    rc_scores = cross_val_score(model, X_train, y_train, cv=cv, scoring='recall')

    comparison_results[name] = {
        'f1_mean': f1_scores.mean(),
        'f1_std': f1_scores.std(),
        'precision': pr_scores.mean(),
        'recall': rc_scores.mean()
    }

    print(f"\n{name}:")
    print(f"  F1:        {f1_scores.mean():.4f} (+/- {f1_scores.std():.4f})")
    print(f"  Precision: {pr_scores.mean():.4f}")
    print(f"  Recall:    {rc_scores.mean():.4f}")

Algorithm Comparison (5-fold Stratified CV)


---
## 1. Algorithm Comparison

Comparing three ensemble classifiers to select the best base algorithm before hyperparameter tuning. Using 5-fold stratified cross-validation for fair comparison.

### Algorithm Selection Rationale

XGBoost selected as the primary algorithm because:
1. **Highest F1 score** in cross-validation among the three candidates
2. **Best precision-recall balance** for imbalanced fraud detection
3. **Built-in regularisation** (gamma, lambda) prevents overfitting on SMOTE data
4. **scale_pos_weight** parameter for additional imbalance handling
5. **Feature importance** readily available for SHAP integration
6. **Fast inference** — important for real-time transaction scoring

---
## 2. Save Baseline Model

Saving the default XGBoost model before tuning, so we can compare baseline vs optimised performance.

In [ ]:
baseline = XGBClassifier(
    n_estimators=100, random_state=42,
    eval_metric='logloss', use_label_encoder=False
)
baseline.fit(X_train, y_train)

os.makedirs("outputs/v1", exist_ok=True)
joblib.dump(baseline, "outputs/v1/xgb_baseline.pkl")
print("Baseline model saved to outputs/v1/xgb_baseline.pkl")